# Regional Inertia Estimation

The Inertia Estimator aggregates generator power, tie-line flow and frequency for one WECC region
(DEMO2's fixed region definitions), then fits a Kalman-filter regression of power deviation on
rate of change of frequency. It publishes:

- `regional_signal`: the regional frequency deviation (pu) and power deviation (MW) for every sample;
- `inertia`: one estimate (mean and median kinetic energy, MVA·s) each time a window of
  `window_sec` seconds closes.

The default template replays one 16 s DEMO2 snapshot (CA1, day 6 hour 1), so the run produces a
single estimate, which is compared with the DEMO2 ground truth below. The estimator's known
limitations are listed in `KNOWN_ISSUES.md` in the `oedisi-inertia` repository.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.feather as pf

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

RUN_DIR = Path(DATA_DIR) if DATA_DIR else Path.cwd()
OUTPUT_DIR = RUN_DIR / "notebook_outputs"

# Payload times count from this instant (ParaEMT's default start; the Player data uses it too).
EPOCH = pd.Timestamp("2026-01-01 00:00:00")

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"

print(f"Run directory: {RUN_DIR}")

## Load the recorded outputs

The run's `wiring.json` says which Recorder is attached to which estimator output, and which
Players feed the estimator, so component names don't matter. Times are converted to seconds.

In [ ]:
def find_run_parts(run_dir: Path):
    """Return the estimator component, {output port: feather path}, and the Players feeding it."""
    wiring = json.loads((run_dir / "wiring.json").read_text(encoding="utf-8"))
    components = {c["name"]: c for c in wiring["components"]}
    estimators = [
        c for c in wiring["components"]
        if {"region", "window_sec"} <= set(c.get("parameters", {}))
    ]
    if len(estimators) != 1:
        raise ValueError(f"Expected one Inertia Estimator in wiring.json; found {len(estimators)}.")
    estimator = estimators[0]

    recordings, players = {}, []
    for link in wiring["links"]:
        if link["source"] == estimator["name"] and components[link["target"]]["type"] == "Recorder":
            recorder = components[link["target"]]
            recordings[link["source_port"]] = (
                run_dir / "build" / recorder["name"] / recorder["parameters"]["feather_filename"]
            )
        elif link["target"] == estimator["name"] and "filename" in components[link["source"]]["parameters"]:
            players.append(components[link["source"]])
    return estimator, recordings, players


def load_output(recordings: dict[str, Path], port: str) -> pd.DataFrame:
    if port not in recordings:
        raise KeyError(f"No Recorder is attached to the estimator's '{port}' output. Add one and run again.")
    path = recordings[port]
    if not path.is_file():
        raise FileNotFoundError(f"{port}: {path} does not exist.")
    frame = pf.read_feather(path)
    frame["seconds"] = (pd.to_datetime(frame["time"]) - EPOCH).dt.total_seconds()
    return frame


estimator, recordings, players = find_run_parts(RUN_DIR)
REGION = estimator["parameters"]["region"]
WINDOW_SEC = float(estimator["parameters"]["window_sec"])

regional = load_output(recordings, "regional_signal")
inertia = load_output(recordings, "inertia")

if regional.empty or not regional["seconds"].is_monotonic_increasing:
    raise ValueError("regional_signal is empty or its time is not monotonic.")
# One regional sample per Player row; the last may be missing if the run stops one step early.
expected_rows = min(int(p["parameters"]["number_of_timesteps"]) for p in players) if players else None
if expected_rows is not None and len(regional) < expected_rows - 1:
    raise ValueError(f"regional_signal has {len(regional)} rows; the Players sent {expected_rows}.")
if inertia.empty:
    raise ValueError(
        f"No inertia estimate was published. The input spans "
        f"{regional['seconds'].iloc[-1] - regional['seconds'].iloc[0]:.4f} s, so a "
        f"{WINDOW_SEC:g} s window never closed (partial windows are discarded)."
    )

print(f"Region: {REGION}; window: {WINDOW_SEC:g} s")
print(f"Regional samples: {len(regional):,}, {regional.seconds.iloc[0]:.4f} to {regional.seconds.iloc[-1]:.4f} s")
print(f"Inertia estimates: {len(inertia)}")

## Regional signal

These are the two series the Kalman filter regresses: power deviation against the rate of change of
frequency deviation. Both are measured from the first sample of the window.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)

axes[0].plot(regional["seconds"], regional["frequency_dev_pu"] * 60e3, color=BLUE, linewidth=1.2)
axes[0].set_ylabel("Frequency deviation (mHz)")
axes[0].set_title(f"{REGION} inertia-weighted frequency deviation", loc="left", color=INK)

axes[1].plot(regional["seconds"], regional["power_dev_mw"], color=BLUE, linewidth=1.2)
axes[1].set_ylabel("Power deviation (MW)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title(f"{REGION} generation plus tie-line power deviation", loc="left", color=INK)

for axis in axes:
    axis.grid(True, alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "regional_signal.png", dpi=200, bbox_inches="tight")
plt.show()

## Estimate vs. ground truth

The ground truth is DEMO2's hourly regional inertia (the H·MBASE sum over the units running in
that hour). It is kept beside the replayed data as `ground_truth.json` and read here only; the
estimator never sees it. If the input isn't a bundled scenario, only the estimate is shown.

In [ ]:
def find_ground_truth(players: list[dict], region: str) -> float | None:
    for player in players:
        path = Path(player["parameters"]["filename"]).expanduser().parent / "ground_truth.json"
        if path.is_file():
            truth = json.loads(path.read_text(encoding="utf-8"))
            if truth.get("region") == region:
                return float(truth["true_energy_mva_s"])
    return None


TRUE_ENERGY = find_ground_truth(players, REGION)

rows = []
for _, estimate in inertia.iterrows():
    for kind in ("mean", "median"):
        value = float(estimate[f"{kind}_energy_mva_s"])
        row = {"window end (s)": estimate["seconds"], "estimate": kind, "energy (MVA·s)": value}
        if TRUE_ENERGY is not None:
            row["ground truth (MVA·s)"] = TRUE_ENERGY
            row["error (MVA·s)"] = value - TRUE_ENERGY
            row["error (%)"] = 100 * (value - TRUE_ENERGY) / TRUE_ENERGY
        rows.append(row)
table = pd.DataFrame(rows)
if TRUE_ENERGY is None:
    print(f"No ground_truth.json for region {REGION} next to the Player data.")
table.style.format(precision=1, thousands=",").format("{:.2f}", subset=["window end (s)"]).hide(axis="index")

In [ ]:
last = inertia.iloc[-1]
labels = ["Mean estimate", "Median estimate"]
values = [float(last["mean_energy_mva_s"]), float(last["median_energy_mva_s"])]

fig, ax = plt.subplots(figsize=(9, 2.8), constrained_layout=True)
bars = ax.barh(labels, values, color=BLUE, height=0.5)
ax.invert_yaxis()
for bar, value in zip(bars, values):
    note = f"{value:,.0f}"
    if TRUE_ENERGY is not None:
        note += f"  ({100 * (value - TRUE_ENERGY) / TRUE_ENERGY:+.1f}%)"
    ax.text(value * 0.98, bar.get_y() + bar.get_height() / 2, note,
            ha="right", va="center", color="white", fontsize=10)
if TRUE_ENERGY is not None:
    ax.axvline(TRUE_ENERGY, color=INK, linestyle="--", linewidth=1.5)
    ax.text(TRUE_ENERGY * 1.01, 0.5, f"Ground truth\n{TRUE_ENERGY:,.0f}", color=INK, va="center", ha="left", fontsize=10)
ax.set_xlim(0, max(values + ([TRUE_ENERGY] if TRUE_ENERGY else [])) * 1.18)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:,.0f}")
ax.set_xlabel("Kinetic energy (MVA·s)")
ax.set_title(f"{REGION} inertia estimate, window ending {last['seconds']:.2f} s", loc="left", color=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="x", alpha=0.25)

fig.savefig(OUTPUT_DIR / "inertia_estimate.png", dpi=200, bbox_inches="tight")
plt.show()

## Reading the result

- The estimate is a mean or median of the filter's per-sample estimates over the whole window,
  including the first second before the disturbance (KNOWN_ISSUES.md #2).
- The power model fits poorly (median R² near zero for this hour, #1), and DEMO2's estimates sit
  10–19% below ground truth for most hours.
- The frequency weights use every machine in the region definition, not only the units running in
  this hour (#5); the ground truth counts only running units.